## AI Startup Launch Advisor
This project uses RAG, chains, an external search tool, and structured output parsing to evaluate startup ideas.

In [18]:
!pip install -q tavily-python transformers accelerate torch langchain langchain-core langchain-classic sentence-transformers faiss-cpu

## 1. Load Libraries and Model

In [19]:
import os
import re
import json
import torch
import faiss
import numpy as np

from tavily import TavilyClient
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

In [22]:
os.environ["TAVILY_API_KEY"] = "PASTE_YOUR_TAVILY_API_KEY_HERE"

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [23]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

if device == "cpu":
    model = model.to(device)

print("Model loaded on:", device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on: cuda


In [25]:
def generate_text(prompt, max_new_tokens=1200):
    messages = [
        {
            "role": "system",
            "content": "You are a startup research and evaluation agent. Return valid JSON when requested."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer.strip()

In [26]:
def extract_json_from_text(text):
    if text is None:
        return None

    text = text.strip()

    if text.lower() in ["null", "none", ""]:
        return None

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    return text[start:end + 1]

## 2. Build RAG Knowledge Base

In [27]:
business_knowledge_base = """
Startup Validation:
A startup idea should be evaluated based on problem clarity, target customer pain point, market demand, solution feasibility, and differentiation from competitors. Strong startup ideas solve urgent problems for a clear customer segment.

Problem-Solution Fit:
Problem-solution fit means that the startup solves a real and important problem for a specific group of users. The problem should be frequent, painful, and valuable enough that users are willing to try or pay for a solution.

Target Customers:
A startup should define its early adopters clearly. Early adopters are users who feel the problem strongly and are most likely to test the first version of the product.

Competitor Analysis:
Competitor analysis compares existing alternatives in the market. Competitors may include direct competitors, indirect competitors, manual solutions, and substitute products. A startup should explain how it is different and why users would switch.

SWOT Analysis:
SWOT stands for Strengths, Weaknesses, Opportunities, and Threats. Strengths and weaknesses are internal factors. Opportunities and threats are external market factors.

MVP Planning:
An MVP is the simplest version of a product that tests the core value proposition. A good MVP should include only the most important features needed to validate the idea with real users.

Business Model:
A business model explains how the startup creates, delivers, and captures value. Common models include subscription, freemium, commission, marketplace, usage-based pricing, licensing, and B2B contracts.

Pricing Strategy:
Early-stage startups should choose simple pricing. Pricing can be based on monthly subscription, pay-per-use, freemium plans, student discounts, or enterprise plans. Pricing should match the customer segment and willingness to pay.

Market Sizing:
Market size can be estimated using TAM, SAM, and SOM. TAM is the total available market, SAM is the serviceable available market, and SOM is the realistic market share the startup can capture at the beginning.

Risk Analysis:
Common startup risks include low willingness to pay, strong competition, weak user retention, technical complexity, poor market timing, legal issues, privacy concerns, and lack of differentiation.

Go-To-Market Strategy:
A startup should define how it will reach its first users. Channels may include social media, university partnerships, direct sales, paid ads, communities, influencers, and referral programs.

Startup Next Steps:
Recommended next steps usually include customer interviews, competitor research, building an MVP, testing with early users, collecting feedback, improving the product, and preparing a launch plan.
"""

In [28]:
def chunk_text(text, chunk_size=120, overlap=20):
    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


business_chunks = chunk_text(
    business_knowledge_base,
    chunk_size=120,
    overlap=20
)

print("Number of RAG chunks:", len(business_chunks))

Number of RAG chunks: 4


## 3. Create FAISS Vector Database

In [29]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

business_embeddings = embedding_model.encode(
    business_chunks,
    convert_to_numpy=True
).astype("float32")

embedding_dim = business_embeddings.shape[1]

rag_index = faiss.IndexFlatL2(embedding_dim)
rag_index.add(business_embeddings)

print("RAG vector database created successfully.")
print("Embeddings shape:", business_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG vector database created successfully.
Embeddings shape: (4, 384)


## 4. Define Chains

In [30]:
def rag_retrieval_chain(startup_idea, industry, target_market, top_k=4):
    query = f"""
    Startup idea: {startup_idea}
    Industry: {industry}
    Target market: {target_market}
    Need: startup validation, SWOT, MVP, pricing, business model, market sizing, competitor analysis
    """

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = rag_index.search(query_embedding, top_k)

    retrieved_chunks = [business_chunks[i] for i in indices[0]]

    business_context = "\n\n".join(retrieved_chunks)

    return business_context, retrieved_chunks

In [32]:
def idea_understanding_chain(startup_idea, industry, target_market):
    idea_profile = {
        "startup_idea": startup_idea.strip(),
        "industry": industry,
        "target_market": target_market,
        "analysis_focus": [
            "problem-solution fit",
            "target customers",
            "competitors",
            "market size",
            "SWOT analysis",
            "business model",
            "MVP plan",
            "pricing strategy",
            "risks",
            "next steps"
        ]
    }

    return idea_profile

In [34]:
def generate_search_queries(startup_idea, industry, target_market):
    prompt = f"""
You are a startup research agent.

Given the startup idea below, generate 5 web search queries:
1. competitors query
2. market size query
3. business model query
4. pricing query
5. industry trend query

Return ONLY a valid JSON object with this structure:
{{
  "competitors_query": "...",
  "market_size_query": "...",
  "business_model_query": "...",
  "pricing_query": "...",
  "trend_query": "..."
}}

Startup idea:
{startup_idea}

Industry:
{industry}

Target market:
{target_market}
"""

    raw_output = generate_text(prompt, max_new_tokens=500)
    json_text = extract_json_from_text(raw_output)

    fallback_queries = {
        "competitors_query": f"{startup_idea} competitors {industry}",
        "market_size_query": f"{industry} market size {target_market}",
        "business_model_query": f"{industry} SaaS business model examples",
        "pricing_query": f"{industry} startup pricing strategy",
        "trend_query": f"{industry} trends {target_market}"
    }

    if json_text is None:
        return fallback_queries

    try:
        return json.loads(json_text)
    except Exception:
        return fallback_queries

## 5. Tavily Web Search Tool

In [35]:
def web_search(query, max_results=5):
    response = tavily_client.search(
        query=query,
        search_depth="advanced",
        max_results=max_results,
        include_answer=True,
        include_raw_content=False
    )

    results = []

    for item in response.get("results", []):
        results.append({
            "title": item.get("title"),
            "url": item.get("url"),
            "content": item.get("content")
        })

    return {
        "query": query,
        "answer": response.get("answer"),
        "results": results
    }

In [36]:
def web_research_chain(startup_idea, industry, target_market):
    queries = generate_search_queries(
        startup_idea=startup_idea,
        industry=industry,
        target_market=target_market
    )

    research_results = {}

    for key, query in queries.items():
        print(f"Searching: {query}")
        research_results[key] = web_search(query, max_results=5)

    return queries, research_results

In [37]:
def format_research_results(research_results, max_items_per_query=3):
    compact_results = {}

    for search_type, data in research_results.items():
        compact_results[search_type] = {
            "query": data.get("query", ""),
            "answer": data.get("answer", ""),
            "results": []
        }

        for item in data.get("results", [])[:max_items_per_query]:
            compact_results[search_type]["results"].append({
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "content": item.get("content", "")[:500]
            })

    return compact_results

## 6. Structured Output Parser

In [38]:
response_schemas = [
    ResponseSchema(
        name="startup_score",
        description="A number from 1 to 10 evaluating the startup idea."
    ),
    ResponseSchema(
        name="overall_judgment",
        description="A short overall judgment about the startup idea."
    ),
    ResponseSchema(
        name="startup_summary",
        description="A short summary of the startup idea."
    ),
    ResponseSchema(
        name="problem",
        description="The main problem the startup solves."
    ),
    ResponseSchema(
        name="solution",
        description="The proposed solution."
    ),
    ResponseSchema(
        name="target_customers",
        description="A list of target customer segments."
    ),
    ResponseSchema(
        name="retrieved_business_context",
        description="A list of important business concepts retrieved from the RAG knowledge base."
    ),
    ResponseSchema(
        name="real_competitors",
        description='A list of competitor objects. Each object must contain: "name", "description", and "source_url".'
    ),
    ResponseSchema(
        name="market_size_analysis",
        description='An object containing: "summary", "evidence", and "source_urls".'
    ),
    ResponseSchema(
        name="swot_analysis",
        description='An object with four fields: "strengths", "weaknesses", "opportunities", and "threats". Each field should be a list.'
    ),
    ResponseSchema(
        name="business_model",
        description="A suggested business model for the startup."
    ),
    ResponseSchema(
        name="mvp_plan",
        description="A list of MVP features or steps."
    ),
    ResponseSchema(
        name="pricing_strategy",
        description="A suggested pricing strategy."
    ),
    ResponseSchema(
        name="risk_analysis",
        description="A list of main risks and challenges."
    ),
    ResponseSchema(
        name="validation_questions",
        description="A list of questions the founder should ask users to validate the idea."
    ),
    ResponseSchema(
        name="next_steps",
        description="A list of recommended next steps."
    ),
    ResponseSchema(
        name="source_links",
        description="A list of source URLs used in the evaluation."
    )
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"startup_score": string  // A number from 1 to 10 evaluating the startup idea.
	"overall_judgment": string  // A short overall judgment about the startup idea.
	"startup_summary": string  // A short summary of the startup idea.
	"problem": string  // The main problem the startup solves.
	"solution": string  // The proposed solution.
	"target_customers": string  // A list of target customer segments.
	"retrieved_business_context": string  // A list of important business concepts retrieved from the RAG knowledge base.
	"real_competitors": string  // A list of competitor objects. Each object must contain: "name", "description", and "source_url".
	"market_size_analysis": string  // An object containing: "summary", "evidence", and "source_urls".
	"swot_analysis": string  // An object with four fields: "strengths", "weaknesses", "opportunities", 

In [40]:
evaluation_prompt_template = PromptTemplate(
    template="""
You are an expert startup evaluator, business analyst, and market research assistant.

You will receive:
1. A startup idea.
2. Business knowledge retrieved from a RAG knowledge base.
3. Real web search results gathered from Tavily.

Your task:
Evaluate the startup idea using:
- the provided RAG business context
- the real web search results
- your business reasoning

Important rules:
- Competitors must come from the provided web search results.
- Market size analysis must mention the source URLs used.
- Do not invent fake competitors.
- If evidence is weak, say that more research is needed.
- The retrieved_business_context field must summarize the RAG context used.
- Return ONLY valid JSON.
- Do not use markdown.
- Do not add text before or after the JSON.

Startup Idea:
{startup_idea}

Industry:
{industry}

Target Market:
{target_market}

RAG Business Context:
{business_context}

Web Research Results:
{research_results}

{format_instructions}
""",
    input_variables=[
        "startup_idea",
        "industry",
        "target_market",
        "business_context",
        "research_results"
    ],
    partial_variables={"format_instructions": format_instructions}
)

In [41]:
def build_fallback_report(startup_idea, industry, target_market, business_context_chunks, research_results):
    competitors_results = research_results.get("competitors_query", {}).get("results", [])
    market_results = research_results.get("market_size_query", {}).get("results", [])

    real_competitors = []

    for item in competitors_results[:5]:
        real_competitors.append({
            "name": item.get("title", "Unknown competitor"),
            "description": item.get("content", "No description available."),
            "source_url": item.get("url", "")
        })

    market_source_urls = []
    market_evidence = []

    for item in market_results[:3]:
        market_source_urls.append(item.get("url", ""))
        market_evidence.append(item.get("content", ""))

    source_links = []

    for search_type, data in research_results.items():
        for item in data.get("results", []):
            url = item.get("url", "")
            if url and url not in source_links:
                source_links.append(url)

    fallback_report = {
        "startup_score": 8,
        "overall_judgment": "The idea appears promising, but it needs validation through customer interviews, MVP testing, and deeper market research.",
        "startup_summary": startup_idea.strip(),
        "problem": "The startup targets a user pain point that may be solved using AI automation.",
        "solution": startup_idea.strip(),
        "target_customers": [
            "Early adopters",
            "Individual users",
            "Small businesses or organizations in the selected industry",
            "Customers in the selected target market"
        ],
        "retrieved_business_context": business_context_chunks,
        "real_competitors": real_competitors,
        "market_size_analysis": {
            "summary": "The market appears attractive based on retrieved web results, but exact market size should be verified using professional market research reports.",
            "evidence": market_evidence,
            "source_urls": market_source_urls
        },
        "swot_analysis": {
            "strengths": [
                "Clear problem-solution direction",
                "Uses AI to automate a repetitive or time-consuming task",
                "Can start with a focused MVP"
            ],
            "weaknesses": [
                "Needs stronger differentiation from competitors",
                "Accuracy depends on model quality and data quality",
                "May need continuous user feedback and iteration"
            ],
            "opportunities": [
                "Growing demand for AI-powered tools",
                "Potential to serve both B2C and B2B customers",
                "Can expand into related features after MVP validation"
            ],
            "threats": [
                "Existing competitors may add similar features",
                "User acquisition may be expensive",
                "Market evidence needs deeper validation"
            ]
        },
        "business_model": "A freemium SaaS model with a free basic plan and paid premium plans for advanced features.",
        "mvp_plan": [
            "Create a landing page explaining the value proposition",
            "Build the core AI feature only",
            "Allow early users to test the product",
            "Collect feedback and usage data",
            "Improve UX and accuracy",
            "Launch a paid beta"
        ],
        "pricing_strategy": "Start with a free plan, then offer monthly subscriptions for premium features. Pricing should be tested with early users.",
        "risk_analysis": [
            "Low willingness to pay",
            "Strong competition",
            "Weak differentiation",
            "Model hallucination or low output quality",
            "Need for reliable data sources"
        ],
        "validation_questions": [
            "How often do users face this problem?",
            "How do users currently solve it?",
            "Would users pay for this solution?",
            "What feature is most important for the MVP?",
            "Why would users choose this product over competitors?"
        ],
        "next_steps": [
            "Interview 20 target users",
            "Analyze the competitors found through web search",
            "Build a simple MVP",
            "Test the MVP with early adopters",
            "Measure retention and willingness to pay",
            "Refine pricing and positioning"
        ],
        "source_links": source_links[:10]
    }

    return fallback_report

## 7. Final Startup Evaluation

In [42]:
def final_evaluation_chain(
    startup_idea,
    industry,
    target_market,
    business_context,
    business_context_chunks,
    compact_research_results
):
    prompt = evaluation_prompt_template.format(
        startup_idea=startup_idea,
        industry=industry,
        target_market=target_market,
        business_context=business_context,
        research_results=json.dumps(
            compact_research_results,
            indent=2,
            ensure_ascii=False
        )
    )

    raw_output = generate_text(prompt, max_new_tokens=1800)

    clean_json = extract_json_from_text(raw_output)

    if clean_json is None:
        print("Model did not return valid JSON. Using fallback structured report.")

        parsed_output = build_fallback_report(
            startup_idea=startup_idea,
            industry=industry,
            target_market=target_market,
            business_context_chunks=business_context_chunks,
            research_results=compact_research_results
        )
    else:
        try:
            parsed_output = output_parser.parse(clean_json)
        except Exception as e:
            print("Parsing failed. Using fallback structured report.")
            print("Error:", e)

            parsed_output = build_fallback_report(
                startup_idea=startup_idea,
                industry=industry,
                target_market=target_market,
                business_context_chunks=business_context_chunks,
                research_results=compact_research_results
            )

    return raw_output, clean_json, parsed_output

In [45]:
def evaluate_startup_agent(startup_idea, industry="AI SaaS", target_market="Global"):
    # Chain 1: Understand startup idea
    idea_profile = idea_understanding_chain(
        startup_idea=startup_idea,
        industry=industry,
        target_market=target_market
    )

    # Chain 2: Retrieve business knowledge using RAG
    business_context, business_context_chunks = rag_retrieval_chain(
        startup_idea=startup_idea,
        industry=industry,
        target_market=target_market,
        top_k=4
    )

    # Chain 3: Web research using Tavily Agent Tool
    queries, research_results = web_research_chain(
        startup_idea=startup_idea,
        industry=industry,
        target_market=target_market
    )

    compact_research_results = format_research_results(
        research_results,
        max_items_per_query=3
    )

    # Chain 4: Final evaluation + Output Parser
    raw_output, clean_json, parsed_output = final_evaluation_chain(
        startup_idea=startup_idea,
        industry=industry,
        target_market=target_market,
        business_context=business_context,
        business_context_chunks=business_context_chunks,
        compact_research_results=compact_research_results
    )

    return {
        "idea_profile": idea_profile,
        "business_context": business_context,
        "business_context_chunks": business_context_chunks,
        "queries": queries,
        "research_results": compact_research_results,
        "raw_output": raw_output,
        "clean_json": clean_json,
        "parsed_output": parsed_output
    }

## 8. Results and Sources

In [46]:
startup_idea = """
An AI-powered platform that helps university students upload lecture recordings,
summarize them, generate study notes, and create quizzes automatically.
"""

result = evaluate_startup_agent(
    startup_idea=startup_idea,
    industry="EdTech",
    target_market="Global"
)

print(json.dumps(result["parsed_output"], indent=4, ensure_ascii=False))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Searching: AI-powered educational platforms for universities - competitors
Searching: Market size of AI-powered education tools for universities globally
Searching: How does an AI-powered platform help university students improve their studies?
Searching: Pricing strategy for an AI-powered platform helping students learn from lectures
Searching: Current trends in AI-driven educational solutions for higher education
{
    "startup_score": 8,
    "overall_judgment": "The startup idea addresses a pressing need for university students by leveraging AI to automate lecture preparation, summarization, note-taking, and quiz generation. The problem is well-defined, and there is a clear market opportunity. However, further validation through user testing and competitive analysis is necessary.",
    "startup_summary": "An AI-powered platform that assists university students in uploading lecture recordings, summarizing them, generating study notes, and creating quizzes.",
    "problem": "Students 

In [47]:
print("Retrieved Business Context from RAG:")
print("=" * 100)

for i, chunk in enumerate(result["business_context_chunks"], 1):
    print(f"\n--- RAG Chunk {i} ---")
    print(chunk)

Retrieved Business Context from RAG:

--- RAG Chunk 1 ---
Startup Validation: A startup idea should be evaluated based on problem clarity, target customer pain point, market demand, solution feasibility, and differentiation from competitors. Strong startup ideas solve urgent problems for a clear customer segment. Problem-Solution Fit: Problem-solution fit means that the startup solves a real and important problem for a specific group of users. The problem should be frequent, painful, and valuable enough that users are willing to try or pay for a solution. Target Customers: A startup should define its early adopters clearly. Early adopters are users who feel the problem strongly and are most likely to test the first version of the product. Competitor Analysis: Competitor analysis compares existing alternatives in the market. Competitors may include direct

--- RAG Chunk 2 ---
startup risks include low willingness to pay, strong competition, weak user retention, technical complexity, poo

In [48]:
print("Generated Search Queries:")
print(json.dumps(result["queries"], indent=4, ensure_ascii=False))

Generated Search Queries:
{
    "competitors_query": "AI-powered educational platforms for universities - competitors",
    "market_size_query": "Market size of AI-powered education tools for universities globally",
    "business_model_query": "How does an AI-powered platform help university students improve their studies?",
    "pricing_query": "Pricing strategy for an AI-powered platform helping students learn from lectures",
    "trend_query": "Current trends in AI-driven educational solutions for higher education"
}


In [49]:
print("Real Competitors / Sources:")
print("=" * 100)

for competitor in result["parsed_output"]["real_competitors"]:
    print("Name:", competitor["name"])
    print("Description:", competitor["description"])
    print("Source:", competitor["source_url"])
    print("-" * 100)

Real Competitors / Sources:
Name: D2L Brightspace
Description: Designed to reimagine how universities create and deliver learning experiences.
Source: https://www.disco.co/blog/ai-lms-platforms-higher-education-2026
----------------------------------------------------------------------------------------------------
Name: Docebo
Description: Enterprise-grade AI support platform for educational institutions.
Source: https://masterofcode.com/blog/best-education-ai-companies
----------------------------------------------------------------------------------------------------
Name: Absorb LMS
Description: Blends robust functionality with user-friendly design.
Source: https://www.absorblms.com/blog/top-ai-learning-platforms
----------------------------------------------------------------------------------------------------


In [50]:
print("Source Links:")
print("=" * 100)

for url in result["parsed_output"]["source_links"]:
    print(url)

Source Links:
https://www.disco.co/blog/ai-lms-platforms-higher-education-2026
https://masterofcode.com/blog/best-education-ai-companies
https://www.absorblms.com/blog/top-ai-learning-platforms
https://www.researchandmarkets.com/reports/5896034/ai-in-education-market-report
https://www.forthecurriculum.com/ai-in-education/


## Conclusion

In this project, an AI Startup Launch Advisor was built using RAG, chains, an external agent tool, and structured output parsing.

The system starts by analyzing the user's startup idea. Then, it retrieves relevant business knowledge from a local RAG knowledge base using embeddings and a FAISS vector database. After that, it uses Tavily Search API as an external search tool to collect real web results about competitors, market size, pricing, and industry trends.

The retrieved RAG context and web search results are passed into a final evaluation chain. Finally, LangChain's StructuredOutputParser is used to generate a clean JSON report containing the startup score, problem, solution, target customers, real competitors, market size analysis, SWOT analysis, business model, MVP plan, pricing strategy, risks, validation questions, next steps, and source links.

This project demonstrates how RAG, tool-using agents, chains, and output parsers can be combined to build a practical AI-powered startup validation system.